In [7]:
import pandas as pd
import zipfile
import os
!pip install pandas


matches_df = pd.read_csv('../ipl_data/ipl_ball_by_ball_2008_2026.csv')

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\p.v.s vinay\AppData\Local\Temp\ipykernel_17372\1197577154.py:7: DtypeWarning: Columns (0: review_by, 1: review_decision) have mixed types. Specify dtype option on import or set low_memory=False.
  matches_df = pd.read_csv('../ipl_data/ipl_ball_by_ball_2008_2026.csv')


In [4]:

kolkata_team = "Kolkata Knight Riders"
opponent_teams = ["Royal Challengers Bengaluru", "Royal Challengers Bangalore", "Chennai Super Kings", "Mumbai Indians"]
home_grounds = ["Eden Gardens", "Chinnaswamy", "Chidambaram", "Wankhede"]

match_records = []

for match_id, group in matches_df.groupby("match_id", sort=False):
    teams = pd.unique(group[["batting_team", "bowling_team"]].to_numpy().ravel())
    teams = [team for team in teams if pd.notna(team)]

    if len(teams) < 2:
        continue

    team1, team2 = teams[:2]
    team_runs = group.groupby("batting_team")["total_runs"].sum()
    team1_runs = int(team_runs.get(team1, 0))
    team2_runs = int(team_runs.get(team2, 0))

    if team1_runs > team2_runs:
        winner = team1
    elif team2_runs > team1_runs:
        winner = team2
    else:
        winner = "Tie"

    match_records.append({
        "match_id": match_id,
        "season": group["season"].iloc[0],
        "date": group["date"].iloc[0],
        "venue": group["venue"].iloc[0],
        "team1": team1,
        "team2": team2,
        "team1_runs": team1_runs,
        "team2_runs": team2_runs,
        "winner": winner
    })

match_summary_df = pd.DataFrame(match_records)

is_kkr_match = (
    ((match_summary_df["team1"] == kolkata_team) & (match_summary_df["team2"].isin(opponent_teams))) |
    ((match_summary_df["team2"] == kolkata_team) & (match_summary_df["team1"].isin(opponent_teams)))
)

filtered_df = match_summary_df[is_kkr_match].copy()


In [5]:


venue_pattern = "|".join(home_grounds)
filtered_df = filtered_df[filtered_df["venue"].str.contains(venue_pattern, case=False, na=False)]

In [6]:

extracted_data = []

for _, row in filtered_df.iterrows():
    if row["team1"] == kolkata_team:
        runs = row.get("team1_runs", 0)
    else:
        runs = row.get("team2_runs", 0)
        
    if row["winner"] == kolkata_team:
        win_loss = "Win"
    elif pd.isna(row["winner"]) or row["winner"] == "Tie":
        win_loss = "Tie/No Result"
    else:
        win_loss = "Loss"
    is_home = "Eden Gardens" in str(row["venue"])  
    extracted_data.append({
        "team": kolkata_team,
        "ground": row["venue"],
        "runs": runs,
        "win/loss": win_loss,
        "is_home": is_home
    })

result_df = pd.DataFrame(extracted_data)
display(result_df)


,team,ground,runs,win/loss,is_home
0,Kolkata Knight Riders,M Chinnaswamy Stadium,222,Win,False
1,Kolkata Knight Riders,"MA Chidambaram Stadium, Chepauk",147,Loss,False
2,Kolkata Knight Riders,Eden Gardens,137,Loss,True
3,Kolkata Knight Riders,Eden Gardens,129,Win,True
4,Kolkata Knight Riders,Wankhede Stadium,67,Loss,False
...,...,...,...,...,...
74,Kolkata Knight Riders,"MA Chidambaram Stadium, Chepauk",107,Win,False
75,Kolkata Knight Riders,Eden Gardens,179,Loss,True
76,Kolkata Knight Riders,Wankhede Stadium,220,Loss,False
77,Kolkata Knight Riders,"MA Chidambaram Stadium, Chepauk",160,Loss,False
